# 🎬 TikTok Money Printer — test en ligne (Colab)

Lance l'app sur une machine gratuite de Google et obtiens une **URL publique temporaire** pour la tester depuis n'importe quel navigateur (même ton téléphone).

**Mode d'emploi :** exécute les 3 cellules dans l'ordre (▶️ à gauche de chaque cellule). À la fin, clique sur l'URL `https://....trycloudflare.com` affichée.

💡 **Astuce vitesse** : menu *Exécution → Modifier le type d'exécution → GPU T4* avant de commencer — la transcription Whisper sera bien plus rapide.

⚠️ L'URL et les fichiers disparaissent quand tu fermes le notebook (session Colab temporaire). Télécharge tes clips au fur et à mesure.

## 1️⃣ Installation (2-3 minutes)

In [ ]:
%cd /content
!rm -rf TIKTOKMONEYPRINTER
!git clone -q -b claude/tiktok-clip-generator-31rslv https://github.com/Kazza2115/TIKTOKMONEYPRINTER.git
%cd TIKTOKMONEYPRINTER
!pip install -q -r requirements.txt
!apt-get -qq install -y ffmpeg > /dev/null
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# vérification : tout doit s'importer sans erreur
import importlib
ok = True
for mod in ['flask', 'yt_dlp', 'faster_whisper', 'anthropic', 'pydantic', 'dotenv']:
    try:
        importlib.import_module(mod)
        print(f'  ✓ {mod}')
    except Exception as e:
        ok = False
        print(f'  ✗ {mod} : {e}')
import shutil
print(f"  {'✓' if shutil.which('ffmpeg') else '✗'} ffmpeg")
print(f"  {'✓' if shutil.which('cloudflared') else '✗'} cloudflared")
print('✅ Installation terminée' if ok else '⚠️ Problème détecté — relance cette cellule, et si ça persiste copie-colle la sortie.')

## 2️⃣ Configuration — ta clé API Claude

⚠️ Choisis un mot de passe **simple : lettres et chiffres uniquement**, sans accents ni espaces (ex: `degzzy2026`).

In [ ]:
import os
from getpass import getpass

key = getpass('🔑 Clé API Anthropic (optionnelle — Entrée pour passer, le mode gratuit marche sans) : ').strip()
if key:
    os.environ['ANTHROPIC_API_KEY'] = key
os.environ['APP_PASSWORD'] = getpass('🔒 Mot de passe de la page (lettres/chiffres, sans accents) : ').strip()
os.environ['WHISPER_MODEL'] = 'small'
os.environ['WHISPER_DEVICE'] = 'auto'  # utilise le GPU automatiquement si activé
print('✅ Configuration enregistrée')
print('   → mode IA disponible' if key else '   → pas de clé API : utilise les modes « gratuit » ou « manuel »')

## 3️⃣ Lancement — clique sur l'URL qui s'affiche

Relançable sans problème : la cellule arrête d'abord toute instance précédente. Si tu changes le mot de passe (cellule 2), relance simplement cette cellule.

In [ ]:
import re
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request

%cd /content/TIKTOKMONEYPRINTER

# stoppe toute instance précédente (serveur + tunnel) pour repartir propre
subprocess.run(['pkill', '-f', 'create_app'], check=False, capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], check=False, capture_output=True)
time.sleep(2)

# préflight : vérifie que l'app s'importe, et affiche l'erreur exacte sinon
check = subprocess.run(
    [sys.executable, '-c', 'from app import create_app; create_app()'],
    capture_output=True, text=True,
)
if check.returncode != 0:
    print('===== ERREUR AU DÉMARRAGE (copie-colle tout ça pour te faire aider) =====')
    print(check.stderr)
    raise RuntimeError("L'app ne démarre pas — l'erreur exacte est affichée ci-dessus.")

# serveur Flask en arrière-plan (hérite du mot de passe défini en cellule 2)
server_log = open('/content/server.log', 'w')
app_proc = subprocess.Popen(
    [sys.executable, '-c', "from app import create_app; create_app().run(host='0.0.0.0', port=5000)"],
    stdout=server_log, stderr=server_log,
)

# attend que le serveur réponde (jusqu'à 30 s)
ready = False
for _ in range(30):
    if app_proc.poll() is not None:
        break
    try:
        urllib.request.urlopen('http://localhost:5000/', timeout=2)
        ready = True
        break
    except urllib.error.HTTPError:
        ready = True  # 401 = le serveur répond (mot de passe actif)
        break
    except Exception:
        time.sleep(1)

if not ready:
    print('===== LOG DU SERVEUR (copie-colle tout ça pour te faire aider) =====')
    server_log.flush()
    print(open('/content/server.log').read() or '(log vide)')
    raise RuntimeError('Le serveur ne répond pas — le log exact est affiché ci-dessus.')

# tunnel public Cloudflare (gratuit, sans compte)
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:5000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
deadline = time.time() + 45
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
    if time.time() > deadline:
        break
# vide la sortie du tunnel en tâche de fond pour ne pas le bloquer
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

if url:
    print('\n' + '=' * 60)
    print(f'🌍 TON APP EST EN LIGNE : {url}')
    print('=' * 60)
    print("\n👤 Identifiant : ce que tu veux (ex: admin)")
    print('🔒 Mot de passe : celui défini à la cellule 2')
    print('Laisse cette cellule tourner tant que tu utilises la page.')
else:
    print('❌ Tunnel non établi — relance cette cellule.')

---
## 🔧 Dépannage

**La popup identifiant/mot de passe boucle sans avancer :**
1. Relance la cellule 2 avec un mot de passe **simple** (lettres/chiffres, sans accents ni espaces)
2. Relance la cellule 3 (elle redémarre le serveur avec le nouveau mot de passe — l'URL change, prends la nouvelle)
3. Dans la popup : identifiant `admin`, puis ton mot de passe

**YouTube bloque le téléchargement** (« Sign in to confirm you're not a bot ») : les IP de Google Cloud sont parfois bloquées. Exporte tes cookies YouTube avec l'extension navigateur *Get cookies.txt LOCALLY*, upload le fichier dans Colab (icône dossier à gauche), puis exécute avant de relancer la cellule 3 :
```python
os.environ['YTDLP_COOKIES'] = open('/content/cookies.txt').read()
```